In [8]:
import json

In [12]:
with open('intent_human_dialogue.json') as f:
    actual = json.load(f)
    
countries_list = ['Austria', 'England', 'France', 'Germany', 'Italy', 'Russia', 'Turkey']

In [13]:
def get_score_dict_from_dialogue(dialogue):
    score_list = []
    empty_predict_count = 0
    
    for item in dialogue:
        if item['intent_dialogue'] == "":
            continue
        
        # 找出所有国家键
        country_keys = [k for k in item.keys() if k in countries_list]
        
        # 过滤掉 '{country}_original_predict' 为空的情况
        valid_countries = []
        for country in country_keys:
            predict_key = f"{country}_original_predict"
            if predict_key in item and item[predict_key] != "":
                valid_countries.append(country)
            else:
                empty_predict_count += 1
        
        # 如果没有有效的国家，则跳过此条记录
        if len(valid_countries) != 2:
            continue
        
        score_list.append(get_score_with_env(valid_countries, item))
    
    return score_list

def get_score_with_env(countries, prediction):
    scores = 0
    for c in countries:
        scores += prediction[c][c.upper()][1]
    score_dict = {}
    score_dict['env'] = prediction['env_uuid']
    score_dict['scores'] = scores / len(countries)
    score_dict['dialogue_count'] = dialogue_count(prediction)
    return score_dict

def dialogue_count(data):
    return len(data['intent_dialogue'].split('\n')) if data['intent_dialogue'] != "" else 0


In [14]:
actual_movement_score_list = get_score_dict_from_dialogue(actual)

In [16]:
len(actual_movement_score_list), len(actual)

(495, 1000)

In [18]:
envs = [i['env'] for i in actual_movement_score_list]

In [21]:
with open('envs.txt', 'w') as f:
    for env in envs:
        f.write(env + '\n')
